In [ ]:
!pip install -r ../../requirements.txt
!pip install -r ../../requirements-dev.txt

# Хуки

## Импорты

In [ ]:
from src.sber.datasets_utils import retrieve_all_datasets, SberDatasetsConfig, unpack_all_datasets
from src.sber.models.extract_features import FeatureExtractorConfig, DummyFeatureModelConfig, DummyFeatureModel
from common.paths import get_sber_gitignore_data_dpath

import torch

from common.logger import JUPYTER_LOGGER as logger

## Загрузка данных

## Dummy

In [2]:
data = retrieve_all_datasets(SberDatasetsConfig())
data = unpack_all_datasets(data)
queries, answers = data

2026-03-28 21:43:19,616 - sber-datasets - INFO - [SBER-DATASETS 📦] Загружаю датасет: rubq-20
2026-03-28 21:43:20,110 - sber-datasets - INFO - [SBER-DATASETS 📦] Найден локальный файл датасета: C:\Users\User\Desktop\dirs\Dev\hack-mfti\sber\data\gitignore\rubq-20\RuBQ_2.0_dev.json
2026-03-28 21:43:31,725 - sber-datasets - INFO - [SBER-DATASETS 📦] Загружаю датасет: tape-chegeka.raw
2026-03-28 21:43:34,732 - sber-datasets - INFO - [SBER-DATASETS 📦] Загружаю датасет: tape-multiq.raw


### Инициализация модели

In [3]:
config = FeatureExtractorConfig(
    probe_layers=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
    enable_moe_routing=True,
    enable_attention_entropy=True,
)
dummy_config = DummyFeatureModelConfig(
    probe_dim=128, vocab_size=100, seed=42
)
dummy_model = DummyFeatureModel(config=config, dummy_config=dummy_config)


def get_ids(text: str):
    return torch.randint(0, dummy_config.vocab_size, (1, 16))

### Извлечение признаков

In [7]:
from tqdm.auto import tqdm

output = []
for query in tqdm(queries, desc="Извлечение признаков"):
    input_ids: torch.Tensor = get_ids(query)
    dummy_output: dict[str, torch.Tensor] = dummy_model(input_ids)
    features = dummy_model.extract(logits=dummy_output["logits"], input_ids=input_ids, answer_start=3)
    out_features = features.to_tensor_dict()
    combined_features = torch.cat(list(out_features.values()), dim=-1)
    output.append(combined_features)

output = torch.stack(output, dim=0)
logger.info(f"Размер итогового тензора: {output.shape}")

Извлечение признаков:   0%|          | 0/31012 [00:00<?, ?it/s]

2026-03-28 21:44:57,472 - jupyter-notebooks - INFO - [JUPYTER 📓] Размер итогового тензора: torch.Size([31012, 233])


### Сохранение признаков

In [9]:
save_dpath = get_sber_gitignore_data_dpath()
save_fpath = save_dpath / "dummy_features.pt"
torch.save(output, save_fpath)
logger.info(f"Признаки сохранены в {save_fpath}")

2026-03-28 21:50:50,755 - jupyter-notebooks - INFO - [JUPYTER 📓] Признаки сохранены в C:\Users\User\Desktop\dirs\Dev\hack-mfti\sber\data\gitignore\dummy_features.pt


### Загрузка признаков

In [10]:
loaded_features = torch.load(save_fpath)
logger.info(f"Загруженные признаки имеют размер: {loaded_features.shape}")

2026-03-28 21:51:23,548 - jupyter-notebooks - INFO - [JUPYTER 📓] Загруженные признаки имеют размер: torch.Size([31012, 233])


## Игра по-взрослому

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from common.paths import get_sber_checkpoints_dpath
from src.sber.models.extract_features import LLMFeatureExtractor

from pathlib import Path

real_model_name: str = "ai-sage/GigaChat3-10B-A1.8B-bf16"
real_model_path: str = str(get_sber_checkpoints_dpath() / real_model_name)

if Path(real_model_path).exists():
    real_model = AutoModelForCausalLM.from_pretrained(real_model_path, torch_dtype=torch.float16, device_map="auto")
    real_tokenizer = AutoTokenizer.from_pretrained(real_model_path)
    logger.info(f"Успешно загрузил модель {real_model_name} из локального кэша")
else:
    logger.warning(f"Локальный путь для модели {real_model_name} не найден, пробую загрузить из Hugging Face")
    real_model = AutoModelForCausalLM.from_pretrained(real_model_name, torch_dtype=torch.float16, device_map="auto")
    real_tokenizer = AutoTokenizer.from_pretrained(real_model_name)
    logger.info(f"Успешно загрузил модель {real_model_name} из Hugging Face")

model = LLMFeatureExtractor.from_default_yaml(model=real_model)
input_text: str = "Вопрос: Сколько будет 2 + 2? Ответ:"
prompt_ids = real_tokenizer.apply_chat_template(
    [{"role": "user", "content": input_text}],
    add_generation_prompt=True,
    tokenize=True,
)
answer_start = len(prompt_ids)

input_ids = torch.tensor([prompt_ids], dtype=torch.long).to(real_model.device)

output = model(input_ids)
features = model.extract(
    logits=output.logits,
    input_ids=input_ids,
    answer_start=answer_start,
)
logger.info(f"Извлеченные признаки имеют размер: {features.to_tensor_dict()['probe_layer_0'].shape}")

In [ ]:
pbar = tqdm(total=len(queries), desc="Извлечение признаков из реальной модели")
real_features = []
for query in queries:
    prompt_ids = real_tokenizer.apply_chat_template(
        [{"role": "user", "content": query}],
        add_generation_prompt=True,
        tokenize=True,
    )
    answer_start = len(prompt_ids)
    input_ids = torch.tensor([prompt_ids], dtype=torch.long).to(real_model.device)
    output = model(input_ids)
    features = model.extract(
        logits=output.logits,
        input_ids=input_ids,
        answer_start=answer_start,
    )
    real_features.append(features.to_tensor_dict())
    pbar.update(1)
pbar.close()
logger.info(f"Извлеченные признаки из реальной модели для всех запросов получены")

In [ ]:
combined_real_features = []
pbar = tqdm(total=len(real_features), desc="Комбинирование признаков")
for feature_dict in real_features:
    combined = torch.cat(list(feature_dict.values()), dim=-1)
    combined_real_features.append(combined)
pbar.close()
final_real_features = torch.stack(combined_real_features, dim=0)
logger.info(f"Итоговый тензор признаков из реальной модели имеет размер: {final_real_features.shape}")

In [ ]:
from common.paths import get_sber_gitignore_data_dpath

save_dpath = get_sber_gitignore_data_dpath()
save_real_fpath = save_dpath / "real_model_features.pt"
torch.save(final_real_features, save_real_fpath)